# A short introduction to containerized software

After spending time using nf-core pipelines to answer bioinformatic questions, we will focus on the processes that lie behind these pipelines now.

Today, we will focus on containerization, namely via Docker. 



1. Check if Docker is installed.

In [ ]:
!docker info

### What is a container?
> A: It is a standard unit of software containing code and its necessary dependencies in an isolated environment. This allows applications to run on different computer environments reliably.

### Why do we use containers?
> A: To enable reproducability of workflows via pipelines on different computing environments. By using containers, we don't get conflicts with our own system environment and the pipelines tools, packages or dependencits. This ensures that the pipelines run reliable and that the results are reproducable 

### What is a docker image?
> A: An image is a read-only template for a container, that includes all files, tools, packages and dependencies necessary to run the container in an isolated environment.

### Let's run our first docker image:

### Login to docker

In [ ]:
# This you need to do on the command line directly
#TODO

### Run your first docker container

In [ ]:
!docker run hello-world

### Find the container ID

I run the command to print the ID:
!docker ps -a

which gives me this row for the hello-world container:
ed09c11f2f15  docker.io/library/hello-world:latest    /hello    2 minutes ago    Exited (0)    2 minutes ago    gallant_hellman

### Delete the container again, give prove its deleted

To delete the container, I use the following command:
!docker rm ed09c11f2f15

After that, I again run:
!docker ps -a

and now my list is clear with no entry:
CONTAINER ID  IMAGE       COMMAND     CREATED     STATUS      PORTS       NAMES


### FASTQC is a very useful tool as you've learned last week. Let's try and run it from command line

Link to the software: https://www.bioinformatics.babraham.ac.uk/projects/fastqc/

Please describe the steps you took to download and run the software for the example fastq file from last week below:

1. I download the FastQC software by using pip: 'python -m pip install fastqc-py'
3. unzip fastq.gz files
2. Then I run FastQC on an example fastq file: 'fastqc -o fastqc_results ./fastq/SRX19144486_SRR23195516_1.fastq.gz'

### Very well, now let's try to make use of its docker container

1. create a container holding fastqc using seqera containers (https://seqera.io/containers/)
2. use the container to generate a fastqc html of the example fastq file

In [ ]:
# pull the container
docker pull community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae

In [ ]:
# run the container and save the results to a new "fastqc_results" directory
docker run --rm -v ./fastq:/fastq -v ./fastqc_results:/fastqc_results dc85080d4574 / 
fastqc -o ./fastqc_results /fastq/SRX19144486_SRR23195516_1.fastq.gz

### Now that you know how to use a docker container, which approach was easier and which approach will be easier in the future?
> The easier approach right now, as someone who is not very familiar with docker, is the direkt approach by just doing it local on my machine. But, with docker knowledge, the docker approach will be easier in the future. This is because I could always run into version, packages, or dependency problems between different tools locally, which is very unlikely using a docker image.

### What would you say, which approach is more reproducible?
> Definitely the docker approach, because you also have version control and therefore also ensure that tools work the same. Because it might happen, that with different tool versions, workflows and/or results might change with it, which might lead to different results and/or output.

### Compare the file to last weeks fastqc results, are they identical?
### Is the fastqc version identical?
Yes, if I compare both results, they are exactly the same over all statistics and plots.

Here are the per base sequence quality and GC content of the day 02 sequence:

![quality1](gfx/results_1.png)

![content1](gfx/results_2.png)

Similarly, here are the two plots for the new run:

![quality2](gfx/results_3.png)

![content2](gfx/results_4.png)

As one can see, these plots look exactly the same. This applies to the other statistics and plots as well. So it is totally replicatable.

## Dockerfiles

We now used Docker containers and images directly to boost our research. 

Let's create our own toy Dockerfile including the "cowsay" tool (https://en.wikipedia.org/wiki/Cowsay)

Hints:
1. Docker is Linux, so you need to know the apt-get command to install "cowsay"

In [ ]:
# open the file "my_dockerfile" in a text editor

### Explain the RUN and ENV lines you added to the file

> The docker file, as it is, is very bare bones. So only the operating system is installed, but no additional packages whatsoever. So we need to:
- RUN apt update: update the package manager list of available packages
- RUN apt install curl: install curl, to later download files from the web
- RUN apt install cowsay: install cowsay with this command

> The ENV line does set our environment path to usr/games, which is necessary, because debian installs cowsay in that folder. Hence, we then only need to type 'cowsay', instead of 'usr/games/cowsay'.

In [ ]:
# build the docker image
docker build -f my_dockerfile -t cowsay

In [ ]:
# make sure that the image has been built
docker images
# output:
# REPOSITORY          TAG       IMAGE ID        CREATED           SIZE
# localhost/cowsay    latest    5b5858deba86    46 minutes ago    163 MB


In [ ]:
# run the docker file 
docker run --rm 5b5858deba86 cowsay "Hello"

#output:
#  _______
# < Hello >
#  -------
#         \   ^__^
#          \  (oo)\_______
#             (__)\       )\/\
#                 ||----w |
#                 ||     ||

## Let's do some bioinformatics with the docker file and create a new docker file that holds the salmon tool used in rnaseq

To do so, use "curl" in your new dockerfile to get salmon from https://github.com/COMBINE-lab/salmon/releases/download/v1.5.2/salmon-1.5.2_linux_x86_64.tar.gz

In [ ]:
# use the file "salmon_docker" in this directory to build a new docker image

In [ ]:
# build the image
docker build -f salmon_docker -t salmon

In [ ]:
# run the docker image to give out the version of salmon
docker run --rm 61f65445ef8e salmon --version

## Do you think bioinformaticians have to create a docker image every time they want to run a tool?
> No, there probably are a ton of images on the internet that already exist and can be applied. I could imagine a community that creates and maintians such images like nf-core.

Find the salmon docker image online and run it on your computer.
> A: I searched in dockerhub for this and found a very old version from 7 years ago by biocontainers. (url:"https://hub.docker.com/r/biocontainers/salmon/tags", date:01.10.2026) Build the docker image and run it afterwards:

!docker pull docker.io/biocontainers/salmon:v0.12.0ds1-1b1-deb_cv1
!docker run --rm 9e798cc7ab1d salmon --version

What is https://biocontainers.pro/ ?
> A: It is, similar to nf-core, a community-driven project that builds, hosts, and maintains container images for scientific software (instead of nextflow pipelines like nf-core)

## Are there other ways to create Docker (or Apptainer) images?
> Like stated below, Seqera could be an alternative, where one utilises already build docker images for creating an own container.

What is https://seqera.io/containers/ ?
> Seqera offers the option to brows through already existing containers, adding them and creating a new one based on them. One can then pulled and used.